In [21]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
import os
from langgraph.graph.message import add_messages 

In [22]:
load_dotenv(find_dotenv())

True

In [23]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [24]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [25]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = model.invoke(messages)
    return {'messages': [response]}

In [ ]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile()

In [28]:
messages = []

while True:
    user_message = input("Type here: ")
    messages.append(HumanMessage(content=user_message))

    print("User: ", user_message)

    if user_message.strip().lower() in ['exit', 'bye', 'quit']:
        break

    response = chatbot.invoke({'messages': messages})
    messages.append(response['messages'][-1])

    print("AI: ", response['messages'][-1].content)

User:  hi
AI:  Hello! How can I assist you today?
User:  what is 5*3?
AI:  5 multiplied by 3 equals 15.
User:  now multiply it by 3.
AI:  15 multiplied by 3 equals 45.
User:  Tell me the complete chat history till now. 
AI:  Certainly! Here’s the chat history so far:

1. User: hi
2. Assistant: Hello! How can I assist you today?
3. User: what is 5*3?
4. Assistant: 5 multiplied by 3 equals 15.
5. User: now multiply it by 3.
6. Assistant: 15 multiplied by 3 equals 45.
7. User: Tell me the complete chat history till now.

Let me know if there's anything else you need!
User:  wow you're smart huh
AI:  Thank you! I'm here to help with any questions or tasks you have. What else would you like to know?
User:  now multiply that number with 6
AI:  45 multiplied by 6 equals 270.
User:  tell me a joke now
AI:  Sure! Here’s a joke for you:

Why did the scarecrow win an award?

Because he was outstanding in his field! 
User:  hahahaha
AI:  I'm glad you enjoyed it! If you want to hear another joke or

In [29]:
messages

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='86a6abf8-db9d-4f46-9a4b-06dffdc3a30f'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 13, 'engine_ttft_ms': 75, 'engine_ttlt_ms': 208, 'pre_inference_ms': 133, 'service_tbt_ms': 18, 'service_ttft_ms': 400, 'service_ttlt_ms': 551, 'total_duration_ms': 451, 'user_visible_ttft_ms': 267}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_27599ce29d', 'id': 'chatcmpl-E6J31lCldBCO4AAJpKpuy1x7XuYOi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs